# Automated Tumor Detection in Whole Slide Images: An End-to-End Deep Learning Pipeline

**A practical guide to building a supervised deep learning system for detecting breast cancer metastases in histopathology images, using the CAMELYON16 challenge dataset.**

---

## Table of Contents

- [1. The Problem: Pathologist Shortages and the Promise of Automation](#section-1)
- [2. The CAMELYON16 Challenge](#section-2)
- [3. Running the code](#section-3)
- [4. Understanding Whole Slide Images](#section-4)
- [5. Step 1: Finding the Tissue](#section-5)
- [6. Step 2: Parsing Tumor Annotations and the Four-Class Labelling Scheme](#section-6)
- [7. Step 3: Building the Training Dataset](#section-7)
- [8. Step 4: The Training Pipeline](#section-8)
- [9. Step 5: Model Architecture](#section-9)
- [10. Step 6: Training](#section-10)
- [11. Step 7: Test Set Evaluation](#section-11)
- [12. Results](#section-12)
- [13. Investigating the Field Cancerisation Hypothesis](#section-13)
- [14. Lessons Learned](#section-14)
- [15. Conclusion](#section-15)

---

<h2 id="section-1">1. The Problem: Pathologist Shortages and the Promise of Automation</h2>

Diagnosing cancer from tissue biopsies remains one of the most critical and labour-intensive tasks in modern medicine. A pathologist examining a sentinel lymph node biopsy must scan an entire tissue section at high magnification, searching for clusters of cancer cells that may occupy only a tiny fraction of the slide. In busy services, pathologists frequently face large daily caseloads under considerable time pressure.

The numbers tell a sobering story. In the UK, the Royal College of Pathologists has warned of a [sustained workforce crisis](https://www.rcpath.org/discover-pathology/public-affairs/the-pathology-workforce.html), with vacancy rates exceeding 30% in some specialties. In the US, the situation is similar: an ageing workforce, rising case volumes, and growing molecular testing demands all strain an already stretched system.

Meanwhile, the digitisation of histopathology is accelerating. Whole Slide Imaging (WSI) scanners now capture tissue sections at resolutions exceeding 100,000 × 100,000 pixels in images that capture cellular-level detail across an entire tissue section. This creates an opportunity: if we can train machine learning models to analyse these images, we can augment pathologists' workflows, flag suspicious regions for closer review, and potentially catch metastases that might be missed under time pressure.

A more speculative research question is whether models can detect subtle alterations in histologically normal-appearing tissue from tumor-bearing slides, potentially reflecting [field cancerisation](https://www.nature.com/articles/nrc.2017.102) or other tumor-associated microenvironmental changes.

In this post, I build an end-to-end pipeline from raw whole-slide images to trained classifiers, and test how far these ideas hold up in practice.

<h2 id="section-2">2. The CAMELYON16 Challenge</h2>

The [CAMELYON16 Grand Challenge](https://camelyon16.grand-challenge.org/) was organised in 2015–2016 by the International Symposium on Biomedical Imaging (ISBI) to benchmark automated detection of breast cancer metastases in whole slide images of sentinel lymph node biopsies.

The dataset consists of nearly **400 H&E-stained whole slide images** from two Dutch medical centres (Radboud UMC and University Medical Centre Utrecht):

| Split | Tumor slides | Normal slides | Total |
|-------|-------------|--------------|-------|
| Train | 110 | 160 | 270 |
| Test  | 49  | 80  | 129 |

Each tumor slide comes with XML annotation files containing polygon outlines of metastatic regions, hand-drawn by expert pathologists.

The winning team [(Wang et al., 2016)](https://arxiv.org/pdf/1606.05718) achieved a slide-level AUC of 0.925 using a GoogLeNet-based patch classifier trained on millions of patches. Remarkably, when the system’s predictions were combined with a pathologist’s review, the pathologist’s AUC increased from 0.966 to 0.995, corresponding to an approximately 85% reduction in error.

My approach follows the same fundamental strategy (patch-based classification) but with one important modification: I introduce a **four-class labelling scheme** that lets me ask more nuanced questions about what the model is capable of actually detecting.

<h2 id="section-3">3. Running the code</h2>

This article shows the full notebook narrative and code, but the project is structured as a modular repository rather than a single standalone notebook. To run it yourself, clone the repo and open the notebook in Google Colab or a local Jupyter environment.

Sections 3–6 can be run directly and download slides from [Amazon S3](https://registry.opendata.aws/camelyon/) (where the full Camelyon16 dataset is openly available) on demand. Sections 10–12 require the pre-generated patch dataset, which you can create using the code in Section 7 (~6–8 hours generation time on Colab) or replace with your own dataset paths.

> **Note**: Training dataset generation and Model training requires Colab Pro (High RAM) to avoid out-of-memory crashes. All other sections run on the free tier.



In [ ]:
# === USER CONFIGURATION ===
# Set these paths before running Sections 10-12.

TRAIN_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_4class_stain_normalised'
TEST_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_test_stain_normalised'

from pathlib import Path

for path, name in [(TRAIN_PATH, 'Training dataset'), (TEST_PATH, 'Test dataset')]:
    if not Path(path).exists():
        raise FileNotFoundError(
            f"{name} not found at: {path}\n\n"
            "Please update the TRAIN_PATH and TEST_PATH variables above to point to your dataset location."
        )

print('Dataset paths verified.')

In [ ]:
# Mount Google Drive and navigate to project folder
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/new_work/Projects/Camelyon16/camelyon16-pathology

!apt-get install -y openslide-tools > /dev/null 2>&1
!pip install -q -r requirements.txt

import numpy as np
import matplotlib.pyplot as plt
import openslide

np.random.seed(42)

# Project imports
from config import DEFAULT_CONFIG
from src.data import list_s3_files, download_file_from_s3, cleanup_file
from src.data.tissue_mask import get_tissue_mask, compute_foreground_mask
from src.data.tumor_polygons import load_tumor_polygons, classify_patch
from src.data.patch_extraction import (
    sample_grid_coordinates, sample_coordinates_by_class,
    extract_patch, preprocess_patch
)
from src.visualisation import (
    visualise_tissue_outline, visualise_patches_grid,
    find_zoom_region_by_coords, find_dense_tissue_region
)

# Example slide used throughout the notebook's tumor-slide walkthrough.
# Changing this allows you to explore a different example normal or tumor slide
EXAMPLE_TUMOR_SLIDE_ID = 'tumor_005'
EXAMPLE_TUMOR_SLIDE = f'{EXAMPLE_TUMOR_SLIDE_ID}.tif'
EXAMPLE_TUMOR_ANNOTATION = f'{EXAMPLE_TUMOR_SLIDE_ID}.xml'

print("All imports successful!")

<h2 id="section-4">4. Understanding Whole Slide Images</h2>

A WSI is not a regular image. At full resolution, a single slide can be 100,000 × 200,000 pixels (roughly **60 gigabytes** of uncompressed pixel data). Given its size, you cannot load one directly into memory.

Instead, WSI formats (like TIFF) use a [**pyramidal structure**](https://camelyon16.grand-challenge.org/Data/): the same image stored at multiple resolutions. I use the [OpenSlide](https://openslide.org/) library to navigate this pyramid, reading small regions on demand without loading the entire file.

Let's download a single slide and explore its structure. 


In [ ]:
# List available slides from S3
all_slides = list_s3_files(DEFAULT_CONFIG.data.s3_images, '.tif')
normal_slides = sorted([f for f in all_slides if 'normal' in f.lower()])
tumor_slides = sorted([f for f in all_slides if 'tumor' in f.lower()])

print(f"Dataset: {len(normal_slides)} normal slides, {len(tumor_slides)} tumor slides")
print(f"\nExample normal slide: {normal_slides[0]}")
print(f"Example tumor slide: {tumor_slides[0]}")

In [ ]:
# Download one tumor slide to explore
slide_name = EXAMPLE_TUMOR_SLIDE
slide_path = download_file_from_s3(
    DEFAULT_CONFIG.data.s3_images, slide_name, '/tmp'
)
slide = openslide.OpenSlide(slide_path)

# Explore the pyramid structure
print(f"Slide: {slide_name}")
print(f"Dimensions (level 0): {slide.dimensions[0]:,} × {slide.dimensions[1]:,} pixels")
print(f"Number of levels: {slide.level_count}")
print(f"\nPyramid levels:")
for i in range(slide.level_count):
    w, h = slide.level_dimensions[i]
    ds = slide.level_downsamples[i]
    print(f"  Level {i}: {w:>7,} × {h:>7,}  (downsample: {ds:.1f}×)")

In [ ]:
# View the whole slide as a thumbnail
thumbnail = slide.get_thumbnail((800, 800))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(thumbnail)
ax.set_title(f'{slide_name}: Thumbnail', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nThe thumbnail is {thumbnail.size[0]}×{thumbnail.size[1]} pixels.")
print(f"The actual slide is {slide.dimensions[0]:,}×{slide.dimensions[1]:,} pixels.")
print(f"That's a {slide.dimensions[0] // thumbnail.size[0]:,}× reduction!")


<h2 id="section-5">5. Step 1: Finding the Tissue</h2>

Most of a whole-slide image is background rather than tissue. Before extracting patches, I need to identify where tissue is present so that sampling is restricted to informative regions.

### Design Decision: Why Work at Thumbnail Resolution?
At full resolution, a single slide may be around 100,000 × 200,000 pixels. For the purpose of locating tissue, however, scanning the full-resolution image is unnecessary: the coarse tissue layout is already visible at much lower magnification. A 512 × 512 thumbnail can be generated quickly by OpenSlide and contains more than enough information for this step.

Crucially, the tissue mask is **never fed to the classifier**. It is used only to identify candidate patch coordinates. Once I have those coordinates in thumbnail space, `get_scaling_factors()` maps them back to the full-resolution image by scaling with the ratio between slide dimensions and mask dimensions:

```
scale_x = slide_width  / mask_width
scale_y = slide_height / mask_height

thumbnail pixel (col, row)
    ↓  ×  scale_x,  ×  scale_y
patch centre (x, y) in level-0 (full resolution) slide coordinates
    ↓  openslide.read_region(location=(x - 112, y - 112), level=0, size=(224, 224))
full-resolution 224 × 224 patch
```

This coarse-detection / fine-extraction workflow is standard in computational pathology. `sample_grid_coordinates()` follows this sequence: iterate over mask pixels, check for tissue, map coordinates back to the full-resolution slide, then extract the patch.

### Design Decision: Why So Many Cleanup Steps?

Greyscale thresholding works in H&E thumbnails because tissue stains pink/purple and is usually **darker than the white glass background**. But a single threshold applied to a raw thumbnail produces a noisy mask with imaging artefacts (such as the border artefacts visible in tumor_005 thumbnail above). `compute_foreground_mask()` applies a deliberate chain of operations to handle each failure mode:

| Step | Operation | What it removes |
|------|----------|-----------------|
| Threshold `< 180` | - | Selects all dark pixels |
| `remove_small_objects()` | `min_size=100` | Dust specks, fibres, staining dots |
| `clear_border()` | - | Any blob touching the image edge (scanner margins are often dark) |
| `remove_small_holes()` | `area_threshold=100` | Small voids within tissue blobs caused by pale staining |
| Border zeroing | `border_margin=5` | Residual edge artefacts that survive `clear_border()` |

`filter_valid_components()` then applies a second filtering pass over the remaining connected components using **two criteria**: minimum area *and* maximum aspect ratio. I include an aspect ratio filter because size alone is not enough: a thin horizontal scanner artefact may have a large pixel area while still having an extreme aspect ratio (for example 10:1), making it unlikely to be genuine tissue. The code rejects any blob exceeding `max_aspect_ratio=5.0`.

**Cleanup order matters.** `clear_border()` must run *before* `filter_valid_components()`. If a border artefact is not removed first, it may be joined to real tissue by a thin pixel bridge and survive as a huge elongated blob.

This sequential process illustrates a broader principle in image segmentation pipelines. Rather than relying on a single operation, these pipelines usually consist of a chain of heuristics, each handling a specific failure mode. The steps taken in `compute_foreground_mask()` and `filter_valid_components()` are a practical response to common WSI artefacts, crafted specifically for this use-case (in my case, via a process of trial and error). 

### Under the Hood: Tissue Masking

The tissue masking logic is surprisingly simple. Here's what `compute_foreground_mask` does - just ~10 lines of core logic:


In [ ]:
# === What compute_foreground_mask does internally ===
from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.segmentation import clear_border

# Step 1: Get a tiny thumbnail (512×512) from the gigapixel image
thumbnail_gray = slide.get_thumbnail((512, 512)).convert("L")
thumbnail_array = np.array(thumbnail_gray)

# Step 2: Simple brightness threshold
# Tissue is darker than the white glass background
threshold = 180  # pixels darker than this are tissue
raw_mask = thumbnail_array < threshold

# Step 3: Morphological cleanup
cleaned = remove_small_objects(raw_mask, min_size=100)   # remove dust specks
cleaned = clear_border(cleaned)                           # remove edge artifacts
cleaned = remove_small_holes(cleaned, area_threshold=100) # fill 'holes' in tissue (likely real tissue)

print(f"Raw mask pixels:     {raw_mask.sum():,}")
print(f"After cleanup:       {cleaned.sum():,}")
print(f"Removed {raw_mask.sum() - cleaned.sum():,} artifact pixels")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(thumbnail_array, cmap='gray')
axes[0].set_title('Grayscale thumbnail')
axes[1].imshow(raw_mask, cmap='gray')
axes[1].set_title(f'After threshold (<{threshold})')
axes[2].imshow(cleaned, cmap='gray')
axes[2].set_title('After morphological cleanup')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Generate tissue mask
mask = compute_foreground_mask(slide)

print(f"Mask shape: {mask.shape}")
print(f"Tissue coverage: {mask.sum() / mask.size:.1%}")

# Visualise: original thumbnail vs. detected tissue
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original
axes[0].imshow(thumbnail)
axes[0].set_title('Original Slide', fontsize=13)
axes[0].axis('off')

# Binary mask
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Tissue Mask', fontsize=13)
axes[1].axis('off')

# Overlay: tissue outline on slide
axes[2].imshow(thumbnail)
# Resize mask to match thumbnail
from PIL import Image
mask_resized = np.array(
    Image.fromarray(mask.astype(np.uint8) * 255).resize(thumbnail.size, Image.NEAREST)
) > 127
axes[2].contour(mask_resized.astype(float), levels=[0.5], colors='lime', linewidths=1.5)
axes[2].set_title('Tissue Detection Overlay', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()


<h2 id="section-6">6. Step 2: Parsing Tumor Annotations and the Four-Class Labelling Scheme</h2>

For tumor slides, CAMELYON16 provides XML files with polygon annotations tracing the tumor boundaries. I parse these into [Shapely](https://shapely.readthedocs.io/) geometry objects, which lets me compute precise overlap between any patch and the annotated tumor regions.

### Design Decision: Labels come from tumor annotation geometry rather than slide identity

For tumor slides, patch labels are assigned from the geometric overlap between the 224 × 224 patch and the annotated tumour regions.
The process inside `classify_patch()`:

1. The patch centre `(x, y)` is converted to a `shapely.box` - a rectangle in level-0 (full resolution) slide coordinate space.
2. `calculate_tumor_overlap()` iterates over all tumor polygons, summing the intersection area between the patch box and each polygon.
3. That total is divided by the patch area (224²) to get an overlap fraction in [0, 1].
4. Threshold rules from `config.py` assign the class label:

```python
if overlap < zero_tolerance:       label = 1  # Normal tissue (tumor slide)
elif overlap < tumor_threshold:    label = 2  # Boundary tumor
else:                              label = 3  # 'Pure' tumor
```

For tumor slides, the label depends on the fraction of the 224 × 224 patch covered by annotated tumour. Patches with essentially zero overlap are labelled normal_from_tumor (class 1), patches with partial overlap are labelled boundary_tumor (class 2), and patches with at least 50% tumour overlap are labelled pure_tumor (class 3). The thresholds are defined explicitly in config.py, which makes the labelling reproducible and auditable.

Class 0 (normal_from_normal) is assigned separately to patches from slides with no tumour annotation file.

### Design Decision: Repairing Invalid Polygon Geometry

Real annotation data is not always perfectly clean. Some CAMELYON16 XML outlines contain invalid or self-intersecting polygon geometry, so `load_tumor_polygons()` repairs these with Shapely before using them in overlap calculations.

### The Four-Class Scheme

The pipeline uses a four-class labelling scheme based on each patch’s spatial relationship to the tumour:

| Class | Name | Description |
|-------|------|-------------|
| 0 | `normal_from_normal` | Normal tissue from a slide with no tumor at all |
| 1 | `normal_from_tumor` | Normal-looking tissue on a slide that also contains tumor |
| 2 | `boundary_tumor` | Tissue at the tumor margin (partial overlap with annotations) |
| 3 | `pure_tumor` | Tissue fully within annotated tumor regions |

Separating classes 0 and 1 is a key scientific choice in this project. Class 1 patches may look very similar to class 0 under the microscope, but they come from tumour-bearing slides. If a model can distinguish them, it may be picking up subtle tumour-associated context, potentially including field cancerisation, stromal response, inflammatory change, or slide-level artefacts. This distinction forms the basis of Experiment 3.

This scheme lets me ask more interesting questions than simple binary classification. In particular, **Class 1 vs. Class 0** tests whether normal-looking tissue from tumor-bearing slides differs detectably from tissue on tumour-free slides.

In [ ]:
# Load tumor annotations
xml_path = download_file_from_s3(
    DEFAULT_CONFIG.data.s3_annotations, EXAMPLE_TUMOR_ANNOTATION, '/tmp'
)
polygons = load_tumor_polygons(xml_path)
print(f"Loaded {len(polygons)} tumor polygons")
for i, p in enumerate(polygons):
    print(f"  Polygon {i}: area = {p.area:,.0f} pixels², "
          f"bounds = {tuple(int(x) for x in p.bounds)}")

In [ ]:
# Visualise tumor annotations overlaid on the tissue
visualise_tissue_outline(
    slide, mask,
    tumor_polygons=polygons,
    title='Tumor Annotations (red) with Tissue Outline (green)',
    figsize=(10, 10)
)


### Classifying Patches by Tumor Overlap

Each patch is classified by computing the **fractional overlap** between the 224×224 patch and the tumor polygons:
- **< 1% overlap** → Class 1 (normal tissue on tumor slide)
- **1–50% overlap** → Class 2 (boundary tissue)
- **≥ 50% overlap** → Class 3 (pure tumor)

The thresholds are configurable, but these defaults ensure clean separation between classes.


### Design Decision: Grid Sampling and Why Stride Is a Modelling Choice

`sample_grid_coordinates()` places a regular grid of candidate patch centres across the tissue mask, separated by a configurable `stride`. This approach is simple, reproducible, and fully auditable - every run with the same seed produces the same coordinates.

**Stride determines overlap and redundancy:**

| Stride | Overlap | Relative patch count |
|--------|---------|----------------------|
| 224 px (= patch size) | None (clean tiling) | 1× |
| 112 px | 50% overlap | 4× |
| 56 px | 75% overlap | 16× |

Smaller stride captures more spatial context and improves coverage of small regions, but introduces **correlation** between nearby patches - consecutive patches at 112 px stride share 50% of their pixels. This matters for evaluation: correlated patches from the same region do not represent independent evidence, so effective sample size is overstated if patches are treated as i.i.d.

**Bounds checking:** `sample_grid_coordinates()` enforces that every patch centre is at least `patch_size // 2 = 112` pixels from each slide edge. This is necessary because patches are *centred* at `(x, y)` rather than anchored at the top-left corner. Without this check, the code would attempt to read off the edge of the slide.


### Design Decision: Class-Specific Sampling Densities

Tumor boundary patches are rare and informationally dense. A slide may have only a thin rim of boundary tissue so sampling at a coarse uniform stride risks missing it almost entirely. `sample_coordinates_by_class()` addresses this by applying **class-specific strides**:

| Class | Default stride | Rationale |
|-------|----------------|-----------|
| Normal (class 1) | 224 px | Abundant everywhere; no overlap needed |
| Pure tumor (class 3) | 112 px | Moderate density for spatial coverage |
| Boundary (class 2) | 56 px | Dense sampling to capture the thin, rare tumor margin |

The implementation samples first at the finest stride (56 px) to classify every tissue coordinate, then subsamples each class independently. The keep ratio is `(target_stride / finest_stride)²` (**squared** because sampling occurs in 2D). A stride ratio of 4 therefore means keeping 1/16 of patches, not 1/4.

> **Why not sample uniformly everywhere?** Because uniform coarse sampling would dramatically undersample the boundary region - the class where the tumor/normal distinction is hardest and most valuable to learn.

The trade-off is that denser boundary sampling increases redundancy and patch-to-patch correlation, so the nominal patch count can overstate the amount of genuinely independent evidence.


### Under the Hood: Patch Classification with Shapely

How do I turn a patch coordinate into a class label? The key is computing the **geometric overlap** between the patch (a square) and the tumor polygons. Here's the core logic:


In [ ]:
# === What classify_patch does internally ===
from shapely.geometry import Polygon, box

# Pick an example coordinate near a tumor boundary
example_coords = sample_coordinates_by_class(slide_path, xml_path)

# Show the classification logic for 3 example patches (one per class)
for class_id in [1, 2, 3]:
    coords = example_coords.get(class_id, [])
    if not coords:
        continue
    x, y = coords[0]  # Take first patch of this class

    # Create a square box for the patch
    patch_size = 224
    half = patch_size // 2
    patch_box = box(x - half, y - half, x + half, y + half)
    patch_area = patch_size * patch_size

    # Compute intersection with ALL tumor polygons
    total_overlap = 0.0
    for polygon in polygons:
        if polygon.intersects(patch_box):
            intersection = polygon.intersection(patch_box)
            total_overlap += intersection.area

    overlap_fraction = min(total_overlap / patch_area, 1.0)

    # Apply classification thresholds
    if overlap_fraction < 0.01:
        label_name = "Class 1: Normal (tumor slide)"
    elif overlap_fraction < 0.50:
        label_name = "Class 2: Boundary"
    else:
        label_name = "Class 3: Pure Tumor"

    print(f"  Patch at ({x}, {y}): overlap = {overlap_fraction:.1%} → {label_name}")


In [ ]:
# Build a regular patch grid over tissue, then classify each patch.
# This preserves the spatial ordering of the grid for visualisation.
grid_stride = 224
coords = sample_grid_coordinates(slide, mask, patch_size=224, stride=grid_stride)

coords_by_class = {1: [], 2: [], 3: []}
for x, y in coords:
    label = classify_patch(x, y, polygons, patch_size=224)
    coords_by_class[label].append((x, y))

print(f"Regular grid stride: {grid_stride} pixels")
print(f"Total tissue patches on grid: {len(coords):,}")
for class_id, class_coords in sorted(coords_by_class.items()):
    class_names = {1: 'Normal (tumor slide)', 2: 'Boundary', 3: 'Pure Tumor'}
    print(f"  Class {class_id} ({class_names[class_id]}): {len(class_coords):,} patches")


In [ ]:
# Visualise the regular patch grid zoomed into the tumor region.
# Center the zoom on pure-tumor patches so the surrounding boundary and normal
# tissue appear in the same ordered grid, as in notebook 02 section 2.3.
zoom_coords = coords_by_class.get(3, []) or (coords_by_class.get(2, []) + coords_by_class.get(1, []))

if zoom_coords:
    zoom_region = find_zoom_region_by_coords(zoom_coords, region_size=10000)
    print(f"Zoom region: {zoom_region}")

    visualise_patches_grid(
        slide,
        coords_by_class,
        zoom_region=zoom_region,
        patch_size=224,
        class_colours={1: 'green', 2: 'orange', 3: 'red'},
        class_labels={
            1: 'Normal',
            2: 'Boundary',
            3: 'Pure Tumor'
        },
        title=f'Zoomed Tumor Region (Grid View) - {slide_name}',
        linewidth=1.5,
        figsize=(14, 12)
    )
else:
    print('No classified patch coordinates were found for visualisation.')


<h2 id="section-7">7. Step 3: Building the Training Dataset</h2>

With my labelling scheme defined, I need to extract hundreds of thousands of patches from hundreds of slides and organise them into a training dataset. Several non-obvious design decisions shape how it works.

### Design Decision: Generate Four Classes, Collapse to Binary at Training Time

The dataset is generated with **four classes**, not two. The class labels are stored in every chunk alongside the patches. At training time, `run_binary_experiment()` remaps them to binary labels depending on the experiment being run.

This separates two concerns: *data collection* (done once, expensively) and *experimental question* (defined cheaply at training time). Generating four classes upfront enables all five binary experiments without re-running the ~6–8 hour extraction pipeline:

| Experiment | Negative class | Positive class |
|------------|----------------|----------------|
| 1: Normal vs Any Tumor | `normal_from_normal` | classes 1, 2, 3 |
| 2: Normal vs Pure Tumor | `normal_from_normal` | `pure_tumor` |
| 3: Slide Context Detection | `normal_from_normal` | `normal_from_tumor` |
| 4: Normal vs Actual Tumor | `normal_from_normal` | classes 2, 3 |
| 5: Normal vs Boundary | `normal_from_normal` | `boundary_tumor` |

For brevity, this blog walks through Experiments 2, 3, and 5 only; Experiments 1 and 4 use the same pipeline and are omitted from the narrative, not from the underlying four-class dataset design.

### Design Decision: Chunk by Slide, Verify Leakage Explicitly

It's not possible to hold all patches in memory simultaneously (> 200 GB uncompressed). Instead, patches are saved in compressed `.npz` chunks of ~1,000 patches each. Each chunk records:

- `X`: patch arrays `(N, 224, 224, 3)`
- `y`: class labels `(N,)`
- `slides`: source slide identifiers `(N,)` - **critical for leakage prevention**
- `coords`: original level-0 coordinates `(N, 2)` - for debugging and visualisation

**Chunking is both a storage decision and a leakage-control mechanism**. WSIs produce many correlated patches. If patches from the same slide appear in both training and validation, the model can memorise slide-specific artefacts such as staining quirks or tissue preparation differences rather than learning genuine pathology. In `FourClassGenerator`, patches are generated one slide at a time and appended to a class-specific buffer. When the buffer reaches the chunk threshold, the entire buffer is saved, so chunks may contain patches from multiple slides and may exceed the nominal chunk_size, but a given slide’s patches **are never split across chunk files**. A chunk-level train/validation split therefore keeps each slide’s patches entirely in one set or the other, while slide IDs stored in the chunk metadata provide an additional explicit leakage check.

> **Dataset generation takes ~6–8 hours** (depending on the number of patches to be generated per class) on Colab and only needs to be run once. The generated dataset is saved to Google Drive for reuse. I provide the generator code below but skip execution - the pre-generated dataset is used for all subsequent steps.

### Design Decision: Stain Normalisation

H&E (haematoxylin and eosin) staining is the foundation of histopathology, but it introduces significant unwanted variation. The same tissue section stained by different labs (or even the same lab on different days) can appear dramatically different in colour due to reagent concentration, staining duration, scanner calibration, and fixation protocols.

This variation is **a technical source of variation that pathologists can usually discount** (i.e. a pathologist can recognise tumor cells regardless of whether the pink is salmon or magenta) but a CNN trained on one lab's colour palette may fail on slides from another. This is one of the central challenges in computational pathology: models that perform well on internal validation often degrade when deployed on external data.

**Stain normalisation** addresses this by transforming every patch to match a reference colour distribution, removing lab-specific variation while preserving diagnostically relevant tissue structure.

Stain normalisation can also suppress colour cues that may be either nuisance variation or biologically meaningful signal, so it should be understood as a bias-variance trade-off rather than an automatic improvement.

### Under the Hood: Macenko Stain Normalisation

I use the [Macenko method](https://ieeexplore.ieee.org/document/5193250) (2009), which models H&E staining as a mixture of two colour components in optical density space. The algorithm:

1. Converts RGB to optical density (where stain concentration has a linear relationship with colour)
2. Uses SVD to identify the two principal stain vectors (H and E) in each image
3. Decomposes patches into "how much haematoxylin" and "how much eosin" at each pixel
4. Rescales these concentrations to match a reference image, then reconstructs

The result preserves tissue structure while standardising colours. I use the [torchstain](https://github.com/EIDOSLAB/torchstain) library, with a manually selected reference patch used consistently across all dataset generation.

In [ ]:
# === Stain Normalisation: Before and After ===
# Demonstrate the effect of Macenko normalisation on patches from different slides

from torchstain.normalizers import MacenkoNormalizer
from PIL import Image
import torch

# Load the reference patch used for all normalisation
reference_path = './data/reference_patch.png'
reference_img = Image.open(reference_path).convert('RGB')
reference_tensor = torch.from_numpy(np.array(reference_img)).permute(2, 0, 1)

# Fit the normalizer to the reference
normalizer = MacenkoNormalizer(backend='numpy')
normalizer.fit(reference_tensor)

# Extract a few patches from different regions of the current slide
# to show stain variation even within a single slide
patch_coords = [
    coords_by_class[1][0] if coords_by_class.get(1) else (50000, 50000),  # Normal region
    coords_by_class[3][0] if coords_by_class.get(3) else (60000, 60000),  # Tumor region
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (x, y) in enumerate(patch_coords):
    # Extract original patch
    patch = slide.read_region((x - 112, y - 112), 0, (224, 224)).convert('RGB')
    patch_array = np.array(patch)

    # Normalise
    patch_tensor = torch.from_numpy(patch_array).permute(2, 0, 1)
    try:
        normalised_tensor, _, _ = normalizer.normalize(patch_tensor)
        normalised_array = normalised_tensor.permute(1, 2, 0).numpy().astype(np.uint8)
        normalisation_success = True
    except Exception as e:
        normalised_array = patch_array  # Fall back to original if normalisation fails
        normalisation_success = False

    # Display
    region_name = "Normal tissue" if row == 0 else "Tumor tissue"

    axes[row, 0].imshow(reference_img)
    axes[row, 0].set_title('Reference patch', fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(patch_array)
    axes[row, 1].set_title(f'{region_name}\n(original)', fontsize=11)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(normalised_array)
    status = '' if normalisation_success else ' (failed)'
    axes[row, 2].set_title(f'{region_name}\n(normalised){status}', fontsize=11)
    axes[row, 2].axis('off')

    # Show the colour distribution shift
    axes[row, 3].hist(patch_array[:,:,0].ravel(), bins=50, alpha=0.5, color='red', label='R (orig)', density=True)
    axes[row, 3].hist(patch_array[:,:,1].ravel(), bins=50, alpha=0.5, color='green', label='G (orig)', density=True)
    axes[row, 3].hist(normalised_array[:,:,0].ravel(), bins=50, alpha=0.5, color='darkred', label='R (norm)', density=True, histtype='step', linewidth=2)
    axes[row, 3].hist(normalised_array[:,:,1].ravel(), bins=50, alpha=0.5, color='darkgreen', label='G (norm)', density=True, histtype='step', linewidth=2)
    axes[row, 3].set_title('Colour distribution shift', fontsize=11)
    axes[row, 3].legend(fontsize=8, loc='upper left')
    axes[row, 3].set_xlabel('Pixel intensity')

plt.suptitle('Macenko Stain Normalisation: Before and After', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nStain normalisation transforms each patch to match the reference colour distribution.")
print("Notice how the normalised patches have more consistent pink/purple tones,")
print("reducing scanner- and slide-specific colour variation.")

In [ ]:
# === DATASET GENERATION (run once, then skip) ===
# This cell is provided for reference. The pre-generated dataset
# is loaded in the next section.

# from src.data.generator import generate_dataset, generate_test_dataset
#
# # Training dataset: ~100K patches per class
# generate_dataset(
#     class_targets={0: 100000, 1: 100000, 2: 100000, 3: 100000},
#     save_path='./data/camelyon16_4class_stain_normalised',
#     stain_normalise=True,
#     reference_image_path='./data/reference_patch.png'
# )
#
# # Test dataset: ~25K patches per class
# generate_test_dataset(
#     class_targets={0: 25000, 1: 25000, 2: 25000, 3: 25000},
#     save_path='./data/camelyon16_test_stain_normalised',
#     stain_normalise=True,
#     reference_image_path='./data/reference_patch.png'
# )

print("Dataset generation code shown above (pre-generated dataset used below)")


In [ ]:
# Verify the dataset paths defined in the user configuration block above.

import os
from pathlib import Path

class_names = {
    0: 'normal_from_normal',
    1: 'normal_from_tumor',
    2: 'boundary_tumor',
    3: 'pure_tumor'
}

print("=== Training Dataset ===")
total_train = 0
for class_id, name in class_names.items():
    class_dir = Path(TRAIN_PATH) / name
    chunks = list(class_dir.glob('*.npz'))
    if chunks:
        sample = np.load(str(chunks[0]))
        n_per_chunk = len(sample['X'])
        sample.close()
        total = len(chunks) * n_per_chunk
        total_train += total
        print(f"  {name}: {len(chunks)} chunks (~{total:,} patches)")

print(f"  Total: ~{total_train:,} patches")

print("\n=== Test Dataset ===")
total_test = 0
for class_id, name in class_names.items():
    class_dir = Path(TEST_PATH) / name
    chunks = list(class_dir.glob('*.npz'))
    if chunks:
        sample = np.load(str(chunks[0]))
        n_per_chunk = len(sample['X'])
        sample.close()
        total = len(chunks) * n_per_chunk
        total_test += total
        print(f"  {name}: {len(chunks)} chunks (~{total:,} patches)")

print(f"  Total: ~{total_test:,} patches")

<h2 id="section-8">8. Step 4: The Training Pipeline</h2>

With my chunked dataset ready, I need a training pipeline that streams patches from chunks without loading everything into memory, remaps 4-class labels to binary for each experiment, prevents slide leakage between splits, and maintains class balance.

### Design Decision: Strictly Class-Balanced Batches

`create_train_dataset()` builds **two separate class-specific streams** (one for the negative class, one for the positive), and forces every batch to draw equally from both:

```python
normal_ds = _create_single_class_dataset(normal_chunks, label=0)
tumor_ds  = _create_single_class_dataset(tumor_chunks,  label=1)

# Half the batch from each class, then concatenate and shuffle positions
dataset = tf.data.Dataset.zip((normal_ds.batch(half_batch), tumor_ds.batch(half_batch)))
          .map(lambda n, t: concat_and_shuffle(n, t))
```

With exact 50/50 balance enforced every batch:

1. **Stable BatchNorm statistics.** If a batch is dominated by one class, batch normalisation learns class-specific running statistics rather than tissue-general ones. This destabilises training.
2. **Consistent gradient updates.** The loss cannot be dominated by the majority class regardless of how chunks happen to be sampled.
3. **Position independence.** `_shuffle_batch()` randomly permutes sample order within each batch so the model cannot learn that "the first 16 samples are always normal".

This improves optimisation stability, but it also changes the effective class prior seen during training: the model is trained on an artificial 50/50 distribution rather than the natural class frequency in the dataset.

### Design Decision: Validation Is Engineered for Stability and Correctness

Training and validation pipelines are built differently on purpose. `create_preloaded_val_dataset()` loads a fixed, class-balanced subset into memory once, interleaves classes at the sample level (N, T, N, T, …), and caches the result. The same data is seen every epoch.

This is a deliberate trade-off: slightly reduced coverage of the validation set in exchange for metrics that are directly comparable epoch-to-epoch.

Without this, validation accuracy oscillated significantly between epochs during development. The instability was partly driven by the validation pipeline, since a generator-based approach yields different batches on each call and injects sampling noise into the metric. 
[SOMETHING ABOUT OTHER SOURCES OF TRAINING INSTABLIITY??]

### Memory Management: The Key Engineering Challenge

Loading the full patch dataset into memory would be impractical. Even as raw `uint8`, 400,000 patches would still occupy about 60 GB in memory. In this pipeline, patches are preprocessed and stored as `float32`, which pushes the raw footprint to roughly 241 GB. The solution is `tf.data.interleave()`, which reads from multiple chunk files simultaneously and yields patches on demand. TensorFlow provides an AUTOTUNE setting for num_parallel_calls, which automatically chooses how many files to process in parallel. In principle this should improve throughput. In practice, for this pipeline it spawned too many workers and led to memory leaks. Replacing it with num_parallel_calls=min(2, cycle_length) - a small explicit cap - made memory usage stable.

### Preventing Slide Leakage

The pipeline splits chunks into train/val sets and then calls `verify_no_slide_leakage()`, which reads the `slides` array from every chunk in both sets and raises a `ValueError` on any overlap. Slide-aware splitting is enforced at the data-generation stage (`FourClassGenerator`) and verified again here at training time.

### Under the Hood: Streaming Chunks with `tf.data`

The training pipeline must feed ~400K patches to the model without loading them all into memory. The core idea is as follows: each chunk file is read on demand using `tf.data.interleave`:

```python
# Simplified version of my chunk reading logic:

def read_chunk(file_path, label):
    with np.load(file_path, mmap_mode="r") as data:
        X = data['X']                     # Memory-mapped, not loaded yet
        idx = np.random.choice(len(X), max_patches, replace=False)
        patches = X[idx].astype(np.float32)  # Only NOW loaded into RAM

        # Normalise to [0, 1]
        if patches.max() > 1.5:
            patches /= 255.0
        patches = np.clip(patches, 0.0, 1.0)

        labels = np.full(len(patches), label, dtype=np.int32)
        return patches, labels

# tf.data.interleave reads from multiple chunks simultaneously,
# yielding a stream of patches without holding everything in memory:
dataset = file_dataset.interleave(
    read_chunk,
    cycle_length=4,           # Read 4 chunks at once
    num_parallel_calls=2,     # CRITICAL: not AUTOTUNE (causes memory leaks)
    deterministic=False       # Allow out-of-order for speed
)
```

**Class balancing** is enforced at the batch level: I create separate streams for each class, batch half from each, then concatenate. This guarantees exactly 50/50 class balance in every batch:

```python
normal_ds = create_class_stream(normal_chunks, label=0)
tumor_ds  = create_class_stream(tumor_chunks,  label=1)

# Each batch: 16 normal + 16 tumor = 32 balanced samples
balanced = tf.data.Dataset.zip((
    normal_ds.batch(16),
    tumor_ds.batch(16)
)).map(lambda n, t: concat(n, t))
```

**Slide leakage prevention**: after splitting chunks into train/val, I read the `slides` array from every chunk and verify zero overlap:

```python
train_slides = collect_slide_ids(train_chunks)
val_slides   = collect_slide_ids(val_chunks)
assert len(train_slides & val_slides) == 0, "Slide leakage detected!"
```


<h2 id="section-9">9. Step 5: Model Architecture</h2>

Convolutional Neural Networks (CNNs) are the natural baseline for this task because histology patches are images, and the relevant signal is spatial. Tumor detection depends on local visual structure such as cell arrangement, tissue texture, and morphology, which convolutional layers are designed to learn directly from pixel data. Rather than treating each pixel independently, they apply small filters across the image to detect visual patterns such as edges, textures, shapes, and more complex structures at increasing levels of abstraction.

I deliberately keep my CNN architectures simple and interpretable, since this project is intended to illustrate the end-to-end pipeline rather than maximise benchmark performance.

### Design Decision: Start Simple, Then Go Subtle

The architecture file provides two primary models. The choice between them depends on the task.

**`simple`**: for visually obvious class differences:
- Larger 5×5 kernels and stride 2 at every layer reduce spatial resolution aggressively.
- Fewer parameters (~66K), fast to train, low overfitting risk.
- Appropriate for Experiment 2 (Normal vs Pure Tumor), where tumor tissue has large-scale morphological differences.

**`subtle`**: for fine-grained tissue analysis:
- Small 3×3 kernels throughout capture finer local structure - individual cell boundaries, nuclear detail, gland architecture.
- The **first layer uses stride 1** rather than stride 2, preserving full spatial resolution for one extra stage before downsampling begins.
- Four convolutional blocks with increasing filter counts (32 → 64 → 128 → 256) give progressively more abstract representations.
- More parameters (~390K), heavier regularisation via dropout at each block.
- Appropriate for harder experiments where differences between classes are subtle or statistical rather than visually obvious.

```
simple:  Input → Conv(16, 5×5, s=2) → Conv(32, 5×5, s=2) → Conv(64, 5×5, s=2) → GAP → Dense(1)   [~66K params]

subtle:  Input → Conv(32, 3×3, s=1) → Conv(64, 3×3, s=2) → Conv(128, 3×3, s=2) → Conv(256, 3×3, s=2) → GAP → Dense(1)   [~390K params]
```

### Design Decision: Global Average Pooling Instead of Flattening

After the final convolutional layer, the feature map is 28 × 28 × 256 for `subtle`. Two options exist for converting this to a classification score:

- **Flatten** → a vector of 28 × 28 × 256 = 200,704 values → one dense layer with ~200K parameters → high overfitting risk, especially on smaller datasets.
- **Global Average Pooling (GAP)** → average each of the 256 feature channels spatially → a 256-dimensional vector → one dense layer with ~256 parameters.

GAP also has a useful inductive bias: it encourages the network to produce activations that are **spatially distributed** across the patch, rather than relying on features at a single specific location. This is particularly appropriate for histology, where diagnostic features (cell nuclei, gland structure, stroma) can appear anywhere within a 224 × 224 patch.

The `Dropout(0.5)` applied after GAP provides additional regularisation immediately before the final sigmoid output, the point in the network where overfitting is most likely to manifest as overconfident predictions.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from src.models.architectures import get_model

# Build and inspect the model
model = get_model('subtle')
model.summary()


<h2 id="section-10">10. Step 6: Training</h2>

I train with the following hyperparameters, chosen through experimentation:
- **Optimiser**: Adam with learning rate `1e-5` (very low - prevents training instability)
- **Gradient clipping**: `clipnorm=1.0` (prevents exploding gradients that cause wild validation oscillations)
- **Callbacks**: ModelCheckpoint (save best val_loss), ReduceLROnPlateau (halve LR after 3 stagnant epochs), EarlyStopping (stop after 3 epochs without improvement)

These settings are a stabilisation choice for this pipeline, not a general recipe: low learning rates and clipping often slow optimisation and may require more epochs to reach a good solution.

> **Note**: Training requires ~6 minutes per epoch on a T4 GPU. The cells below execute the full training runs. If you want to skip training, pre-trained models can be loaded from the `models/` directory.


In [ ]:
# Configure training
from src.models import run_binary_experiment
from config import DEFAULT_CONFIG

DEFAULT_CONFIG.training.normalise_patches = False
DEFAULT_CONFIG.training.val_max_samples_per_class = 4000

# Uses TRAIN_PATH defined in Section 7 above
TRAIN_DATASET_PATH = TRAIN_PATH

# ============================================================
# EXPERIMENT 2: Normal vs Pure Tumor (sanity check - should be easy)
# ============================================================
print("=" * 60)
print("EXPERIMENT 2: Normal vs Pure Tumor")
print("=" * 60)

exp2_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=2,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp2_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp2_results['results']['auc']:.3f}")


In [ ]:
# ============================================================
# EXPERIMENT 5: Normal vs Boundary (harder)
# ============================================================
print("=" * 60)
print("EXPERIMENT 5: Normal vs Boundary Tumor")
print("=" * 60)

exp5_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=5,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp5_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp5_results['results']['auc']:.3f}")


In [ ]:
# ============================================================
# EXPERIMENT 3: Slide Context Detection (field cancerisation hypothesis)
# ============================================================
print("=" * 60)
print("EXPERIMENT 3: Slide Context Detection")
print("=" * 60)

exp3_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=3,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp3_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp3_results['results']['auc']:.3f}")

<h2 id="section-11">11. Step 7: Test Set Evaluation</h2>

Validation metrics are computed on held-out chunks from the same pool of training slides. The true test of generalisation is the **held-out test set**, which uses entirely different slides not seen during training or validation.

### Design Decision: Pick the Decision Threshold from Validation Data

A sigmoid output is a score, not a calibrated class probability. The default threshold of 0.5 is arbitrary - it is only optimal if the model is perfectly calibrated *and* false positives and false negatives are equally costly. Neither holds in general.

Instead, `find_optimal_threshold()` selects the threshold that maximises **Youden's J statistic** on the validation predictions:

```
J = sensitivity + specificity − 1
  = TPR − FPR
```

Youden's J is maximised at the point on the ROC curve where the trade-off between catching true positives and avoiding false positives is best balanced. The implementation uses scikit-learn's `roc_curve()` to compute J at every candidate threshold and returns the best:

```python
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
j_scores = tpr - fpr
best_threshold = thresholds[np.argmax(j_scores)]
```

**This threshold is selected on validation data only, then held fixed for test evaluation.** Using test labels to select or adjust the threshold would be a form of evaluation leakage since the test set would no longer be truly held out. The threshold is saved in the model's JSON metadata file (`save_model_metadata()`) so it is retrieved consistently whenever the model is loaded:

```python
meta = load_model_metadata('./models/normal_vs_pure_tumor.keras')
threshold = meta['threshold']                     # From validation - not re-fitted
predictions = (test_scores >= threshold).astype(int)
```

In [ ]:
# Test set evaluation
from src.models import evaluate_on_test_set, load_model_metadata

# Uses TEST_PATH defined in Section 7 above

# Load models and metadata
experiments_to_eval = {
    'exp2': {
        'name': 'Normal vs Pure Tumor',
        'model_path': './models/normal_vs_pure_tumor.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['pure_tumor']},
        'results': exp2_results
    },
    'exp5': {
        'name': 'Normal vs Boundary',
        'model_path': './models/normal_vs_boundary.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['boundary_tumor']},
        'results': exp5_results
    },
    'exp3': {
        'name': 'Slide Context Detection',
        'model_path': './models/slide_context_detection.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['normal_from_tumor']},
        'results': exp3_results
    }
}

test_results = {}
for key, exp in experiments_to_eval.items():
    print(f"\n{'='*60}")
    print(f"TEST: {exp['name']}")
    print(f"{'='*60}")

    model = keras.models.load_model(exp['model_path'])
    meta = load_model_metadata(exp['model_path'])
    normalise = meta.get('normalise_patches', False)

    result = evaluate_on_test_set(
        model, TEST_PATH, exp['mapping'], key,
        threshold=meta['threshold'],
        normalise=normalise
    )
    test_results[key] = result

    print(f"Val AUC:  {exp['results']['results']['auc']:.3f}")
    print(f"Test AUC: {result['auc']:.3f}")
    print(f"Gap:      {exp['results']['results']['auc'] - result['auc']:.3f}")
    print(result['report'])


<h2 id="section-12">12. Results</h2>

The table below summarises performance across the three experiments. These are canonical results from a representative run; your results may differ slightly (see note below).

| Experiment | Task | Val AUC | Test AUC | Test Accuracy |
|------------|------|---------|----------|---------------|
| Exp 2 | Normal vs Pure Tumor | 0.870 | 0.838 | 78.3% |
| Exp 5 | Normal vs Boundary | 0.727 | 0.633 | 58.0% |
| Exp 3 | Slide Context Detection | 0.627 | 0.494 | 48.8% |

### Interpretation

**Experiment 2 (Normal vs Pure Tumor)** achieves strong performance with a test AUC of 0.838. This is expected: pure tumor tissue has visibly different morphology — disrupted gland architecture, irregular nuclei, and increased cell density — making this task relatively straightforward for a CNN. While this falls short of the 0.925 AUC achieved by the CAMELYON16 winners, those approaches used much larger models (GoogLeNet with ImageNet pretraining), millions of patches, and sophisticated post-processing pipelines. My lightweight CNN trained from scratch on ~400K patches performs respectably as a baseline.

**Experiment 5 (Normal vs Boundary)** is harder, with test AUC dropping to 0.633. Boundary patches contain a mix of normal and tumor tissue, often with only subtle invasion at the margins. The 0.09 gap between validation and test suggests the model learns something real but struggles to generalise fully — possibly because boundary appearance varies more between slides than pure tumor.

**Experiment 3 (Slide Context Detection)** performs at chance on the test set (AUC 0.494), despite above-chance validation performance (0.627). This is the key negative result discussed in Section 13: the validation signal likely reflects slide-level confounds rather than genuine field cancerisation.

### A Note on Run-to-Run Variation

Each training run may produce slightly different results due to:
- **Random weight initialisation**: neural networks start from different random states
- **Batch sampling order**: even with seeded generators, `tf.data` pipelines introduce non-determinism when `num_parallel_calls > 1`
- **Chunk selection**: which chunks are assigned to train vs validation varies with shuffling
- **GPU non-determinism**: cuDNN optimisations can produce slightly different floating-point results

In [ ]:
# Summary comparison chart
exp_names = ['Exp 2: Normal vs\nPure Tumor', 'Exp 5: Normal vs\nBoundary', 'Exp 3: Slide\nContext']
exp_keys = ['exp2', 'exp5', 'exp3']

val_aucs = [experiments_to_eval[k]['results']['results']['auc'] for k in exp_keys]
test_aucs = [test_results[k]['auc'] for k in exp_keys]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(exp_names))
width = 0.35

# AUC comparison
bars1 = axes[0].bar(x - width/2, val_aucs, width, label='Validation', color='steelblue')
bars2 = axes[0].bar(x + width/2, test_aucs, width, label='Test', color='darkorange')
axes[0].set_ylabel('AUC', fontsize=12)
axes[0].set_title('Validation vs Test AUC', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_names, fontsize=10)
axes[0].legend(fontsize=11)
axes[0].set_ylim(0.4, 1.0)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
for i, (v, t) in enumerate(zip(val_aucs, test_aucs)):
    axes[0].text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(i + width/2, t + 0.02, f'{t:.3f}', ha='center', fontsize=9, fontweight='bold')

# Val-Test gap
gaps = [v - t for v, t in zip(val_aucs, test_aucs)]
colours = ['forestgreen' if g < 0.05 else 'darkorange' if g < 0.1 else 'firebrick' for g in gaps]
axes[1].bar(x, gaps, color=colours, width=0.5)
axes[1].set_ylabel('AUC Gap (Val - Test)', fontsize=12)
axes[1].set_title('Generalisation Gap', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_names, fontsize=10)
axes[1].axhline(y=0, color='black', linewidth=0.5)
for i, g in enumerate(gaps):
    axes[1].text(i, g + 0.005, f'{g:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


<h2 id="section-13">13. Investigating the Field Cancerisation Hypothesis</h2>

The most scientifically interesting result is Experiment 3: can a model detect whether normal-looking tissue came from a slide that also contains a tumor?

My initial validation results suggested a weak but detectable signal (AUC ~0.64). However, evaluation on the held-out test set showed performance near random chance (AUC ~0.54). To ensure this wasn't a model capacity issue, I tested four architectures of increasing complexity:

| Architecture | Val AUC | Test AUC | Trainable Params |
|---|---|---|---|
| `subtle` (custom CNN) | 0.585 | 0.543 | ~390K |
| `attention` (spatial attention) | 0.610 | 0.467 | ~390K |
| `transfer` (frozen MobileNetV2) | 0.504 | - | ~1.3K |
| `transfer_finetune` (fine-tuned MobileNetV2) | 0.585 | - | ~700K |

The consistency across architectures is telling: model capacity is likely not the limiting factor. The weak validation signal likely reflects **slide-level confounds** (staining batch effects, tissue preparation differences) rather than genuine field cancerisation. These confounds are shared between training and validation slides (from the same pool) but don't transfer to the test slides.

### What Would It Take?

Properly testing this hypothesis would require approaches beyond patch-level classification:
- **Multi-instance learning**: aggregate evidence across many patches per slide, rather than classifying patches independently
- **Pathology foundation models**: use feature extractors pre-trained on millions of pathology images (e.g., UNI, CONCH) rather than ImageNet
- **Slide-level prediction**: treat each slide as a single example, using all its patches collectively

This is itself a valuable finding: a clean negative result that establishes what *doesn't* work and points toward what might.

<h2 id="section-14">14. Lessons Learned</h2>

### Technical Lessons

1. **Preprocessing consistency is critical.** My most dramatic bug was a mismatch between training and test preprocessing: models saw `[0, 1]` scaled data during training but raw `[0, 255]` data during evaluation. Validation AUC was 0.93; test AUC was 0.53. Always have a single source of truth for preprocessing transformations.

2. **Memory management dominates development time.** Whole slide images, patch datasets, and TensorFlow data pipelines all compete for RAM. Carefully managing memory during dataset generation and training (setting `num_parallel_calls=2` instead of `AUTOTUNE`, limiting validation set size, and above all using generators instead of loading data into memory) is essential. `AUTOTUNE` can keep increasing parallel chunk reads to chase throughput until RAM is saturated, whereas an explicit value caps the number of concurrent workers and makes memory use predictable.

3. **Stabilising training and validation was the hardest engineering challenge**. For weeks, validation metrics oscillated so violently that it was often unclear whether the model was improving or the pipeline was just noisy. Fixing this required stabilising two things at once: the model updates and the validation data. A low learning rate (1e-5) and gradient clipping (clipnorm=1.0) reduced weight instability, while a deterministic validation set removed batch-to-batch sampling noise. Stain normalisation was equally important, since inconsistent colour distributions destabilised the entire pipeline. This reflects a real-world pathology problem: models trained on one site's staining and scanning conditions often degrade when deployed on data from another.

4. **Start with the easiest experiment.** I used Normal vs Pure Tumor (the visually obvious case) to validate the entire train→evaluate→test pipeline before investing GPU time on harder tasks. This caught the preprocessing bug early.

### Scientific Lessons

5. **Negative results are results.** My field cancerisation experiment didn't find a generalisable signal, despite trying four architectures. This is informative: it constrains what's possible with patch-level H&E classification and motivates alternative approaches.

6. **Validation ≠ generalisation.** Even with no slide leakage, validation and test performance can diverge significantly when the task is hard. Validation slides share distributional properties with training slides; test slides may not.

7. **Know when to stop.** When four architectures all converge on the same weak result, the bottleneck is likely the data or the task, not the model. Throwing more complexity at a data-limited problem is unlikely to help.

<h2 id="section-15">15. Conclusion</h2>

I built a complete pipeline for automated tumor detection in histopathology images: from gigapixel whole slide images through tissue detection, patch extraction, stain normalisation, and CNN-based classification. My models achieve **0.87–0.89 test AUC** for detecting tumor tissue - a useful screening tool, though short of the 0.925 achieved by the CAMELYON16 winners (who used larger models and more data, as well as some clever tricks like hard negative mining where the model is re-trained on false-positive patches to reduce these false alarm errors). 

The field cancerisation hypothesis that normal tissue near tumors carries detectable molecular changes remains unresolved by my patch-level approach. The signal I observed in validation did not survive the test set, suggesting that more sophisticated methods (multi-instance learning, pathology foundation models) are needed.

The full codebase, including all modules, configurations, and notebooks, is available on [GitHub](https://github.com/balintstewart77/camelyon16-pathology).

---
